# 05 — Measure key events

Measure local-baseline-corrected positive, negative, peak-to-peak, and reduced pressure for the key arrivals.

In [1]:

from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from project_config import ensure_output_dirs

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [2]:

from obspy import read
from event_measurements import (
    measure_reduced_pressures_in_window,
    pressure_results_for_paper,
)

st_corr = read(str(DERIVED_DIR / "bchh_corrected_analysis_window.pkl"),
               format="PICKLE")
geometry = pd.read_csv(DERIVED_DIR / "bchh_geometry.csv")
SENSOR_DISTANCES_M = (
    geometry.loc[geometry["channel"].str.startswith("HD")]
    .set_index("channel")["distance_m"]
    .to_dict()
)


## Measurement windows

In [3]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")

event_windows = [
    {
        "event": "Initial second-stage failure",
        "start": EXPLOSION_TIME + 3.0 - 0.08,
        "end": EXPLOSION_TIME + 5.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.80,
        "signal_start_s": 0.85,
        "signal_end_s": 1.40,
    },
    {
        "event": "Principal explosion",
        "start": EXPLOSION_TIME + 6.0 - 0.08,
        "end": EXPLOSION_TIME + 9.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.75,
        "signal_start_s": 0.75,
        "signal_end_s": 2.50,
    },
    {
        "event": "Capsule pulse 1",
        "start": UTCDateTime("2016-09-01T13:07:28.30"),
        "end": UTCDateTime("2016-09-01T13:07:28.75"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.12,
        "signal_start_s": 0.12,
        "signal_end_s": 0.45,
    },
    {
        "event": "Capsule pulse 2",
        "start": UTCDateTime("2016-09-01T13:07:28.85"),
        "end": UTCDateTime("2016-09-01T13:07:29.25"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.10,
        "signal_start_s": 0.10,
        "signal_end_s": 0.40,
    },
]


In [4]:

all_results = []
event_streams = {}

for spec in event_windows:
    event_stream, result = measure_reduced_pressures_in_window(
        st_corr,
        spec["start"],
        spec["end"],
        SENSOR_DISTANCES_M,
        reference_distance_m=1000.0,
        event_name=spec["event"],
        baseline_start_s=spec["baseline_start_s"],
        baseline_end_s=spec["baseline_end_s"],
        signal_start_s=spec["signal_start_s"],
        signal_end_s=spec["signal_end_s"],
    )
    event_streams[spec["event"]] = event_stream
    all_results.append(result)

key_event_pressures = pd.concat(all_results, ignore_index=True)
display(pressure_results_for_paper(key_event_pressures))
key_event_pressures.to_csv(
    DERIVED_DIR / "key_event_pressure_measurements.csv",
    index=False,
)


,event,channel,distance_m,positive_peak_pa,negative_peak_pa,peak_to_peak_pa,positive_reduced_pa,negative_reduced_pa,peak_to_peak_reduced_pa
0,Initial second-stage failure,HD1,1442.2,29.2,-67.1,96.3,42.1,-96.7,138.9
1,Initial second-stage failure,HD2,1410.2,52.9,-21.8,74.7,74.6,-30.7,105.3
2,Initial second-stage failure,HD3,1414.8,48.4,-15.2,63.7,68.5,-21.5,90.1
3,Initial second-stage failure,MEDIAN,1414.8,48.4,-21.8,74.7,68.5,-30.7,105.3
4,Principal explosion,HD1,1442.2,1334.8,-193.3,1528.1,1925.1,-278.8,2203.9
5,Principal explosion,HD2,1410.2,1545.9,-230.6,1776.5,2180.0,-325.2,2505.2
6,Principal explosion,HD3,1414.8,1423.3,-234.1,1657.4,2013.6,-331.3,2344.9
7,Principal explosion,MEDIAN,1414.8,1423.3,-230.6,1657.4,2013.6,-325.2,2344.9
8,Capsule pulse 1,HD1,1442.2,240.9,-112.8,353.7,347.4,-162.7,510.1
9,Capsule pulse 1,HD2,1410.2,279.1,-115.4,394.6,393.6,-162.8,556.4
